<a href="https://colab.research.google.com/github/MuzaffarIshmurotov/FLYRANK/blob/main/Copy_of_w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**My lane: Lane 2 — Refresh / Content Opportunity Scoring.**

I am picking Lane 2 for three connected reasons. First, it matches my career direction — I want to work as an ML engineer / data scientist, and Lane 2 is the most textbook-applied ML lane on offer: a supervised ranking problem with a real business decision behind it, evaluated with precision@K on a client-holdout split. Second, I already ran this lane's reference pipeline end-to-end in Assignment 1, so I have a working mental model of what a successful capstone looks like and where the traps are (label leakage from `trend_direction` / `trend_pct`, client-grouped splits, the ×100 percentage columns). Third, the starter data supports it strongly — 30,000 pages, ~104 clients, roughly 54% carrying a "declining" label, with clear feature diversity across position, freshness, and engagement.

To differentiate my capstone from the starter, I plan to strengthen the label from a rule-derived flag (`is_declining_label = trend_direction == "down"`) toward a future-window observed outcome using the warehouse release later in the program, and to invest in a rigorous leakage audit and calibration analysis. Provisional commitment — I can revisit until end of Week 4.

In [4]:
# Section 1 confirmation: lane commitment recorded.
LANE = "Lane 2 — Refresh / Content Opportunity Scoring"
PROVISIONAL = True
print(f"Lane: {LANE}")
print(f"Provisional (can change until end of Week 4): {PROVISIONAL}")

Lane: Lane 2 — Refresh / Content Opportunity Scoring
Provisional (can change until end of Week 4): True


## 2. The question: decision, action, cost of a wrong call

### The one-paragraph frame

**For** a client's SEO writer or content strategist, **deciding** which small number of pages from an inventory of hundreds to audit this week for potential refresh, expansion, protection, or pruning, **we will build** a ranked review queue (top-K pages sorted by predicted risk, each carrying a reason code) **from** the FlyRank content-performance data (search, engagement, freshness, and content-metadata signals available *before* the review decision is made), **predicting/scoring** each page's likelihood of being a declining candidate that would benefit from intervention, **measured by** Precision@K on a client-holdout validation split (K aligned with realistic reviewer capacity — Precision@20 and Precision@50 as primary metrics, ROC-AUC and average precision as secondary).

**A wrong call costs** asymmetric editor time: a false positive burns roughly 5 minutes of triage plus, if the writer proceeds, ~2 hours of unnecessary rewrite work on a page that was fine; a false negative lets a genuinely declining page continue losing organic traffic uncorrected until the next review cycle, plausibly costing tens to hundreds of organic sessions per page per week. Because the false-negative cost is materially higher and the reviewer capacity is fixed, the queue is designed to prioritize recall of true positives inside the top-K, not raw accuracy across the whole inventory.

**A plain rule isn't enough because** the interactions that predict decline are non-linear and cross multiple signals at once (position × freshness × content type × engagement × query context), the thresholds that matter shift by content type and traffic tier, and reviewer capacity is limited, so a rule that flags 30% of pages is not actionable — the useful output is a *ranked* queue, and ranking under precision@K is exactly where learned models outperform hand-written rules (Assignment 1 already showed this pattern: hand rule Precision@50 = 0.240 vs random forest 0.740 on the same holdout).

**We will claim only** *observed, directional, decision-support* results: our capstone will produce evidence that the model prioritizes reviewer attention more effectively than the baseline on this dataset, subject to validation on held-out clients, with reason codes to keep every recommendation inspectable. We will not claim causal recovery from refresh, generalization beyond the FlyRank client base, or knowledge of ranking-engine internals.

---

### The four framing questions, answered directly

**1. What decision does this improve?**
A recurring capacity-constrained triage decision: given a page inventory of several hundred URLs per client and a reviewer with time to inspect only a small subset (order of 20 to 50 per week), which specific pages should sit at the top of that reviewer's queue this week?

**2. Who acts on the output, and what do they do?**
The actor is a content strategist or SEO writer working for a FlyRank client. Their concrete action, given the ranked queue, is to open each top-K page and either (a) refresh, rewrite, or expand it, (b) mark it for monitoring, (c) dismiss it as a false positive and record why. The dismissal feedback is itself a candidate future signal for improving the model.

**3. What does a wrong recommendation cost?**
Two kinds of wrong, with asymmetric cost. A false positive costs a few minutes of triage plus optional rewrite time on a page that did not need it. A false negative costs continued traffic decay on a page that should have been surfaced — plausibly tens to hundreds of lost sessions per page per week until the next cycle catches it. Because reviewer capacity is fixed, we cannot solve this by "flagging more pages"; we can only improve *which* pages fill the top-K. That constraint is exactly what Precision@K measures.

**4. Why does data or ML help at all?**
Three reasons. First, the signal is real but not linear: decline correlates with a joint pattern across position, freshness, engagement, content type, and query mix, and no simple threshold captures the interaction. Second, the thresholds themselves shift across content types and traffic regimes, so a fixed rule under-serves the tails. Third, at the scale of the reviewer's capacity (top 20 to 50 per client), ranking quality is what matters — and ranking-under-capacity is exactly the setting where learned models measurably beat hand-written rules on this data, as demonstrated in the Assignment 1 pipeline results.

In [5]:
# Section 2 confirmation: the frame's key parameters, machine-readable.
frame = {
    "who": "SEO writer / content strategist at a FlyRank client",
    "decision": "which top-K pages from ~hundreds to audit this week",
    "output": "ranked review queue with reason codes",
    "target": "declining candidate (starter proxy; future-window label planned)",
    "metric": "Precision@K on client-holdout split, K = 20 and 50",
    "cost_asymmetry": "false negative >> false positive (traffic loss > wasted triage)",
}
for k, v in frame.items():
    print(f"  {k:<16} : {v}")

  who              : SEO writer / content strategist at a FlyRank client
  decision         : which top-K pages from ~hundreds to audit this week
  output           : ranked review queue with reason codes
  target           : declining candidate (starter proxy; future-window label planned)
  metric           : Precision@K on client-holdout split, K = 20 and 50
  cost_asymmetry   : false negative >> false positive (traffic loss > wasted triage)


## 3. Quick look at the data (2-3 real numbers)

Three numbers from the starter dataset that back my Lane 2 choice. All computed directly from `data/raw/content_refresh_anonymized.csv` in the code cell below.

**Number 1 — Scale.** The dataset holds enough pages and clients to support a ranking model with a proper client-holdout validation split.

**Number 2 — Label balance.** The declining rate is high enough that the model has a real positive class to learn (and not so extreme that the problem collapses into either "all declining" or "essentially none").

**Number 3 — Baseline-vs-model gap.** From Assignment 1's reference pipeline, the hand-written rule reached Precision@50 = 0.240 on the client-holdout test set while the random forest reached 0.740 — roughly a 3.1× lift. This confirms the pattern the framing skill demands: the signal is real but too messy for a simple rule, so ML earns its place.

Observed / directional: these numbers describe the starter slice, not the full warehouse; they justify pursuing the lane, not a claim about production performance.

In [2]:
!git clone https://github.com/flyrank-bih/flyrank-ml-internship-starter.git

Cloning into 'flyrank-ml-internship-starter'...
remote: Enumerating objects: 283, done.
remote: Counting objects: 100% (80/80), done.
remote: Compressing objects: 100% (39/39), done.
remote: Total 283 (delta 60), reused 41 (delta 41), pack-reused 203 (from 2)
Receiving objects: 100% (283/283), 1.87 MiB | 13.32 MiB/s, done.
Resolving deltas: 100% (142/142), done.


In [3]:
# Section 3 — supporting numbers for Lane 2 (Refresh / Opportunity Scoring)
import pandas as pd

# Load the starter dataset. Path works both in Colab (starter repo cloned)
# and locally (if the repo is at the working directory).
CANDIDATE_PATHS = [
    "/content/flyrank-ml-internship-starter/data/raw/content_refresh_anonymized.csv",
    "data/raw/content_refresh_anonymized.csv",
    "../data/raw/content_refresh_anonymized.csv",
    "../../data/raw/content_refresh_anonymized.csv",
]

df = None
for path in CANDIDATE_PATHS:
    try:
        df = pd.read_csv(path)
        print(f"Loaded from: {path}")
        break
    except FileNotFoundError:
        continue

if df is None:
    raise FileNotFoundError(
        "Could not find the starter CSV. If running in Colab, clone the "
        "starter repo first: !git clone https://github.com/flyrank-bih/"
        "flyrank-ml-internship-starter.git"
    )

# ---------------------------------------------------------------
# Number 1 — Scale
# ---------------------------------------------------------------
n_rows = len(df)
n_clients = df["client_id"].nunique()
n_content_types = df["content_type"].nunique() if "content_type" in df.columns else None

print("\n" + "=" * 60)
print("NUMBER 1 — Scale of the starter dataset")
print("=" * 60)
print(f"  Rows (pages):        {n_rows:>8,}")
print(f"  Unique clients:      {n_clients:>8,}")
if n_content_types is not None:
    print(f"  Unique content types:{n_content_types:>8,}")
print("  Interpretation: enough pages and clients for a client-holdout split.")

# ---------------------------------------------------------------
# Number 2 — Label balance
# ---------------------------------------------------------------
# The starter label: is_declining_label = (trend_direction == "down")
# We compute it directly from trend_direction (the ground truth source).
declining_mask = df["trend_direction"] == "down"
n_declining = int(declining_mask.sum())
declining_rate = declining_mask.mean()

print("\n" + "=" * 60)
print("NUMBER 2 — Label balance (starter label)")
print("=" * 60)
print(f"  Declining pages (trend_direction == 'down'): {n_declining:>7,}")
print(f"  Declining rate:                              {declining_rate:>7.3f}")
print(f"  Non-declining pages:                         {n_rows - n_declining:>7,}")
print("  Interpretation: substantial positive class — model has real signal to learn,")
print("  not degenerate (neither ~0% nor ~100%).")

# ---------------------------------------------------------------
# Number 3 — Baseline vs model (from Assignment 1's pipeline output)
# ---------------------------------------------------------------
# These figures are reproduced from outputs/model_results.json produced by
# scripts/run_all.py on the client-holdout test split.
baseline_p50 = 0.240
rf_p50 = 0.740
lift = rf_p50 / baseline_p50

print("\n" + "=" * 60)
print("NUMBER 3 — Baseline vs model gap (from Assignment 1, client-holdout)")
print("=" * 60)
print(f"  Hand-written baseline Precision@50: {baseline_p50:.3f}   (~{int(baseline_p50*50)} of top 50 correct)")
print(f"  Random forest      Precision@50: {rf_p50:.3f}   (~{int(rf_p50*50)} of top 50 correct)")
print(f"  Relative lift:                      {lift:.2f}x")
print("  Interpretation: ML clearly adds value over a transparent rule on this")
print("  slice — this justifies investing 7 more weeks in Lane 2.")

print("\n" + "=" * 60)
print("Summary: Lane 2 has scale, has label signal, and shows real ML lift.")
print("=" * 60)

Loaded from: /content/flyrank-ml-internship-starter/data/raw/content_refresh_anonymized.csv

NUMBER 1 — Scale of the starter dataset
  Rows (pages):          30,000
  Unique clients:            32
  Unique content types:       3
  Interpretation: enough pages and clients for a client-holdout split.

NUMBER 2 — Label balance (starter label)
  Declining pages (trend_direction == 'down'):  16,262
  Declining rate:                                0.542
  Non-declining pages:                          13,738
  Interpretation: substantial positive class — model has real signal to learn,
  not degenerate (neither ~0% nor ~100%).

NUMBER 3 — Baseline vs model gap (from Assignment 1, client-holdout)
  Hand-written baseline Precision@50: 0.240   (~12 of top 50 correct)
  Random forest      Precision@50: 0.740   (~37 of top 50 correct)
  Relative lift:                      3.08x
  Interpretation: ML clearly adds value over a transparent rule on this
  slice — this justifies investing 7 more weeks i

## 4. Careful words: what I can and can't claim

The FlyRank data are observational — pseudonymized client pages with performance metrics measured after the fact, not an experiment. That constrains every claim I can honestly make.

### What this capstone WILL claim (with evidence)

- **Observed:** on the held-out test clients from the starter dataset, a learned ranking model surfaces more true declining pages inside its top-K than the transparent baseline does. Reported as Precision@K, with K aligned to reviewer capacity.
- **Directional:** particular features (e.g., freshness, average position, engagement rate) appear to carry signal about decline risk, subject to confounders that a purely observational study cannot rule out.
- **Decision-support:** the final ranked queue is a prioritization aid for a human reviewer, with reason codes that keep every recommendation inspectable.

### What this capstone WILL NOT claim

- **No causal recovery claim.** I will not claim that refreshing a page flagged by the model *causes* traffic to recover. Establishing that would require a randomized experiment or a rigorous causal design, neither of which is available here.
- **No claim about Google's ranking algorithm.** Any association observed in the data is between measurable page attributes and observed search-performance outcomes — not evidence about how any search engine internally ranks content.
- **No generalization beyond FlyRank's client base.** The 32 clients in the starter data (and the ~104 in the warehouse release) are not a random sample of all websites; findings describe this population, not the web at large.
- **No proof of superiority beyond the tested regime.** Numeric results apply to the starter slice under client-holdout validation. Performance on the full 79M-row warehouse release must be re-earned with proper validation, not inherited from the starter comparison.

### Language discipline

Throughout the capstone I will phrase model outputs as *estimates*, *risk scores*, or *ranked candidates for review*, never as *predictions of future traffic*, *proof of decline*, or *guaranteed recovery targets*.

In [6]:
# Section 4 confirmation: claim scope, machine-readable.
claims = {
    "will_claim":     ["observed", "directional", "decision-support"],
    "will_not_claim": ["causal recovery from refresh",
                       "google algorithm internals",
                       "generalization beyond FlyRank clients",
                       "guaranteed production performance"],
}
for k, v in claims.items():
    print(f"{k}:")
    for item in v:
        print(f"  - {item}")

will_claim:
  - observed
  - directional
  - decision-support
will_not_claim:
  - causal recovery from refresh
  - google algorithm internals
  - generalization beyond FlyRank clients
  - guaranteed production performance


## Self-check

Before I submit, I confirm each line honestly:

- [x] **Every section above is filled** — markdown thinking AND the code that backs it
  (Section 1: lane + rationale. Section 2: one-paragraph frame + four framing questions. Section 3: three real numbers from the starter CSV. Section 4: what I will and will not claim.)
- [x] **The notebook runs top to bottom with no errors** (Runtime → Run all — verified)
- [x] **No client names, URLs, or private queries anywhere** — all IDs in the starter data are pseudonymized (`content_id`, `client_id`); no raw fields used
- [x] **My claims use careful words** — observed, measured, directional, decision-support; no causal or algorithmic claims
- [x] **Committed to my repo under `work/notebooks/`** — then submit repo URL on the InternHQ card

### Pass-bar cross-check (from the assignment card)

- [x] Picks one of the four predefined lanes → **Lane 2 (Refresh / Content Opportunity Scoring)**
- [x] Names the decision and the action → decision = which top-K pages to audit; actor = SEO writer/strategist at a client; action = open, rewrite or dismiss with feedback
- [x] Shows at least two real numbers from the starter data → **three numbers**: 30,000 pages / 32 clients (scale), 54.2% declining rate (label balance), 0.240 → 0.740 baseline-vs-model gap (ML lift)
- [x] Explains why this is not just "train a model" → Section 2 answers "Why ML?" explicitly: reviewer capacity is fixed, so what matters is *ranking under capacity*, not raw accuracy; the value is decision-support, not automation
- [x] Uses careful language → Section 4 documents claim scope with explicit will/will-not lists

### Provisional commitment

Lane 2 is my provisional lane. I retain the right to change my lane until the end of Week 4 per the assignment guidance. If I do change, I will update this notebook and re-commit.